# Augmentation Quality Evaluation: CTGAN vs TabDDPM

This notebook evaluates the fidelity of synthetic data generated by CTGAN and tabDDPM
against the corresponding real data for both the **Balanced** and **Per-Class** augmentation strategies.

## Metrics computed (state-of-the-art tabular data fidelity suite)

### Per-feature statistical metrics
- **Kolmogorov-Smirnov (KS) statistic** per numeric feature — marginal distribution distance
- **Wasserstein-1 (Earth Mover's) distance** per numeric feature — distributional cost
- **Mean / Std difference** per numeric feature
- **One-hot activation rate difference** (TVD) per binary feature

### Global dataset-level metrics
- **Average KS statistic** across all numeric features
- **Average Wasserstein distance** across all numeric features
- **Correlation matrix Frobenius distance** (real vs synthetic correlation structures)
- **Total Variation Distance (TVD)** on one-hot features
- **PCA Coverage Ratio** — fraction of real PCA variance explained by synthetic samples
- **Pairwise MMD (Maximum Mean Discrepancy)** — kernel-based distributional distance

### Outputs
- Per-run JSON metrics file
- Heatmaps: KS statistic per feature per class
- Heatmaps: Wasserstein distance per feature per class
- Heatmaps: One-hot TVD per feature per class
- Correlation matrix comparison plots (real vs synthetic)
- PCA projection overlays
- Summary comparison: Balanced vs Per-Class for each model
- Global summary CSV comparing all four configurations


In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import json
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from scipy.stats import ks_2samp, wasserstein_distance
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)

print("Imports OK")

Imports OK


In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────

REPO_ROOT = Path(".")   # adjust if running from a different working directory

CTGAN_ROOT   = REPO_ROOT / "5G-NIDD_Aug_CTGAN"
TABDDPM_ROOT = REPO_ROOT / "5G-NIDD_Aug_tabDDPM"

OUT_ROOT = REPO_ROOT / "augmentation_eval"

LABEL_COL   = "Attack Type"
RANDOM_SEED = 42
PCA_SAMPLE  = 10_000   # rows sampled per source for PCA plots
HIST_BINS   = 50

# Attack classes present in the dataset
ATTACK_CLASSES = [
    "Benign", "HTTPFlood", "ICMPFlood", "SYNFlood",
    "SYNScan", "SlowrateDoS", "TCPConnectScan", "UDPFlood", "UDPScan"
]

# Numeric features to focus on for distributions
PREFERRED_NUM = [
    "Dur", "SrcPkts", "DstPkts", "SrcBytes", "DstBytes",
    "sMeanPktSz", "Load", "SrcLoad", "DstLoad", "Loss"
]

# Create output directories
for model in ["ctgan", "tabddpm"]:
    for strategy in ["balanced", "perclass"]:
        (OUT_ROOT / model / strategy).mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "comparison").mkdir(parents=True, exist_ok=True)

print("Output directories created under:", OUT_ROOT)

Output directories created under: augmentation_eval


In [3]:
# ── Helper utilities ──────────────────────────────────────────────────────────

def read_csv_safe(path):
    """Read CSV, returning None if the file is missing or a git-lfs pointer."""
    p = Path(path)
    if not p.exists():
        return None
    if p.stat().st_size < 500:          # LFS pointer files are tiny
        with open(p) as f:
            first = f.read(30)
        if "git-lfs" in first or "version https" in first:
            print(f"  [SKIP LFS] {p.name}")
            return None
    try:
        return pd.read_csv(p, engine="python", on_bad_lines="skip")
    except Exception as e:
        print(f"  [ERROR] {p}: {e}")
        return None


def detect_feature_types(df, label_col=None):
    """Return (num_cols, bin_cols) excluding label."""
    exclude = {label_col} if label_col else set()
    bin_cols, num_cols = [], []
    for c in df.columns:
        if c in exclude:
            continue
        if not pd.api.types.is_numeric_dtype(df[c]):
            continue
        u = set(df[c].dropna().unique())
        if u <= {0, 1, 0.0, 1.0}:
            bin_cols.append(c)
        else:
            num_cols.append(c)
    return num_cols, bin_cols


def pick_numeric_features(df, num_cols):
    """Pick up to 10 highest-variance numeric features, prioritising preferred list."""
    preferred = [c for c in PREFERRED_NUM if c in num_cols]
    remaining = [c for c in num_cols if c not in preferred]
    top = preferred + list(
        df[remaining].var().sort_values(ascending=False).head(10 - len(preferred)).index
    )
    return top[:10]


def compute_ks(real_col, syn_col):
    r = real_col.dropna().values
    s = syn_col.dropna().values
    if len(r) == 0 or len(s) == 0:
        return np.nan
    stat, _ = ks_2samp(r, s)
    return float(stat)


def compute_wasserstein(real_col, syn_col):
    r = real_col.dropna().values
    s = syn_col.dropna().values
    if len(r) == 0 or len(s) == 0:
        return np.nan
    return float(wasserstein_distance(r, s))


def compute_tvd_binary(real_col, syn_col):
    """Total variation distance for a binary feature."""
    r_rate = float(real_col.fillna(0).mean())
    s_rate = float(syn_col.fillna(0).mean())
    return abs(r_rate - s_rate)


def corr_frobenius(real_df, syn_df, cols):
    """Frobenius norm of (corr_real - corr_syn) for given columns."""
    common = [c for c in cols if c in real_df.columns and c in syn_df.columns]
    if len(common) < 2:
        return np.nan
    cr = real_df[common].corr().fillna(0).values
    cs = syn_df[common].corr().fillna(0).values
    return float(np.linalg.norm(cr - cs, "fro"))


def compute_mmd(X_real, X_syn, gamma=1.0, n=2000):
    """RBF-kernel MMD estimate (subsampled for speed)."""
    rng = np.random.default_rng(RANDOM_SEED)
    r = X_real[rng.choice(len(X_real), min(n, len(X_real)), replace=False)]
    s = X_syn[rng.choice(len(X_syn),  min(n, len(X_syn)),  replace=False)]

    def rbf(A, B):
        diff = A[:, None, :] - B[None, :, :]
        return np.exp(-gamma * (diff ** 2).sum(-1))

    mmd = rbf(r, r).mean() + rbf(s, s).mean() - 2 * rbf(r, s).mean()
    return float(max(0.0, mmd))


def pca_coverage(real_df, syn_df, feat_cols, n_components=2):
    """
    Fit PCA on real data; project both. Return fraction of real PC bounding
    box that is covered by synthetic samples (per component).
    """
    common = [c for c in feat_cols if c in real_df.columns and c in syn_df.columns]
    if len(common) < 2:
        return np.nan
    scaler = StandardScaler()
    R = scaler.fit_transform(real_df[common].fillna(0))
    S = scaler.transform(syn_df[common].fillna(0))
    pca = PCA(n_components=n_components, random_state=RANDOM_SEED)
    R_p = pca.fit_transform(R)
    S_p = pca.transform(S)
    # Coverage: mean fraction of real percentile range [5,95] that syn covers
    coverages = []
    for i in range(n_components):
        lo, hi = np.percentile(R_p[:, i], [5, 95])
        if hi == lo:
            continue
        in_range = ((S_p[:, i] >= lo) & (S_p[:, i] <= hi)).mean()
        coverages.append(float(in_range))
    return float(np.mean(coverages)) if coverages else np.nan


print("Helpers defined.")

Helpers defined.


In [4]:
# ── Core evaluation function ──────────────────────────────────────────────────

def evaluate_pair(real_df, syn_df, label=None, feat_cols=None):
    """
    Evaluate one (real, synthetic) pair.
    Returns a dict of all scalar metrics + per-feature dicts.
    """
    real_df = real_df.copy()
    syn_df  = syn_df.copy()

    # Align columns
    common_cols = [c for c in real_df.columns if c in syn_df.columns and c != LABEL_COL]
    real_df = real_df[common_cols]
    syn_df  = syn_df[common_cols]

    num_cols, bin_cols = detect_feature_types(real_df)

    if feat_cols is None:
        feat_cols = pick_numeric_features(real_df, num_cols)

    metrics = {
        "label": label,
        "real_rows": len(real_df),
        "syn_rows":  len(syn_df),
        "n_features": len(common_cols),
        "ks_per_feature": {},
        "wasserstein_per_feature": {},
        "mean_diff_per_feature": {},
        "std_diff_per_feature": {},
        "tvd_per_binary_feature": {},
    }

    # Per numeric-feature metrics
    for c in num_cols:
        if c not in syn_df.columns:
            continue
        rc = pd.to_numeric(real_df[c], errors="coerce")
        sc = pd.to_numeric(syn_df[c],  errors="coerce")
        metrics["ks_per_feature"][c]          = compute_ks(rc, sc)
        metrics["wasserstein_per_feature"][c]  = compute_wasserstein(rc, sc)
        metrics["mean_diff_per_feature"][c]    = abs(float(rc.mean()) - float(sc.mean()))
        metrics["std_diff_per_feature"][c]     = abs(float(rc.std())  - float(sc.std()))

    # Per binary-feature TVD
    for c in bin_cols:
        if c not in syn_df.columns:
            continue
        metrics["tvd_per_binary_feature"][c] = compute_tvd_binary(real_df[c], syn_df[c])

    # Aggregate scalar metrics
    ks_vals  = [v for v in metrics["ks_per_feature"].values() if not np.isnan(v)]
    wd_vals  = [v for v in metrics["wasserstein_per_feature"].values() if not np.isnan(v)]
    tvd_vals = [v for v in metrics["tvd_per_binary_feature"].values() if not np.isnan(v)]

    metrics["avg_ks"]           = float(np.mean(ks_vals))  if ks_vals  else np.nan
    metrics["avg_wasserstein"]  = float(np.mean(wd_vals))  if wd_vals  else np.nan
    metrics["avg_tvd_binary"]   = float(np.mean(tvd_vals)) if tvd_vals else np.nan

    # Correlation Frobenius distance
    metrics["corr_frobenius"] = corr_frobenius(real_df, syn_df, feat_cols)

    # PCA coverage
    metrics["pca_coverage"] = pca_coverage(real_df, syn_df, feat_cols)

    # MMD (on scaled numeric features)
    num_common = [c for c in feat_cols if c in real_df.columns and c in syn_df.columns]
    if len(num_common) >= 2:
        scaler = StandardScaler()
        R_sc = scaler.fit_transform(real_df[num_common].fillna(0))
        S_sc = scaler.transform(syn_df[num_common].fillna(0))
        metrics["mmd"] = compute_mmd(R_sc, S_sc)
    else:
        metrics["mmd"] = np.nan

    return metrics


print("evaluate_pair() defined.")

evaluate_pair() defined.


In [5]:
# ── Plotting helpers ──────────────────────────────────────────────────────────

def plot_correlation_comparison(real_df, syn_df, feat_cols, title, out_path):
    common = [c for c in feat_cols if c in real_df.columns and c in syn_df.columns]
    if len(common) < 2:
        return
    cr = real_df[common].corr()
    cs = syn_df[common].corr()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    short = [c[:8] for c in common]

    sns.heatmap(cr, ax=axes[0], cmap="coolwarm", vmin=-1, vmax=1,
                xticklabels=short, yticklabels=short, square=True, annot=False)
    axes[0].set_title("Real correlations")

    sns.heatmap(cs, ax=axes[1], cmap="coolwarm", vmin=-1, vmax=1,
                xticklabels=short, yticklabels=short, square=True, annot=False)
    axes[1].set_title("Synthetic correlations")

    diff = cr.values - cs.values
    v = max(abs(diff).max(), 0.01)
    sns.heatmap(pd.DataFrame(diff, index=common, columns=common),
                ax=axes[2], cmap="RdBu_r", vmin=-v, vmax=v,
                xticklabels=short, yticklabels=short, square=True, annot=False)
    axes[2].set_title("Difference (Real - Synthetic)")

    fig.suptitle(title, fontsize=13, y=1.02)
    plt.tight_layout()
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close(fig)


def plot_pca_overlay(real_df, syn_df, feat_cols, title, out_path, label_col=None):
    common = [c for c in feat_cols if c in real_df.columns and c in syn_df.columns]
    if len(common) < 2:
        return
    scaler = StandardScaler()
    R = scaler.fit_transform(real_df[common].fillna(0))
    S = scaler.transform(syn_df[common].fillna(0))
    pca = PCA(n_components=2, random_state=RANDOM_SEED)
    R_p = pca.fit_transform(R)
    S_p = pca.transform(S)

    n = PCA_SAMPLE
    rng = np.random.default_rng(RANDOM_SEED)
    ri = rng.choice(len(R_p), min(n, len(R_p)), replace=False)
    si = rng.choice(len(S_p), min(n, len(S_p)), replace=False)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(R_p[ri, 0], R_p[ri, 1], s=4, alpha=0.4, label="Real", color="steelblue")
    ax.scatter(S_p[si, 0], S_p[si, 1], s=4, alpha=0.3, label="Synthetic", color="darkorange")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
    ax.set_title(title)
    ax.legend(markerscale=3)
    plt.tight_layout()
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close(fig)


def plot_distribution_grid(real_df, syn_df, feat_cols, title, out_path):
    cols = [c for c in feat_cols if c in real_df.columns and c in syn_df.columns][:10]
    if not cols:
        return
    n = len(cols)
    ncols = 5
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 2.8))
    axes = np.array(axes).flatten()

    for i, c in enumerate(cols):
        r = pd.to_numeric(real_df[c], errors="coerce").dropna()
        s = pd.to_numeric(syn_df[c],  errors="coerce").dropna()
        axes[i].hist(r, bins=HIST_BINS, alpha=0.55, density=True,
                     label="Real", color="steelblue")
        axes[i].hist(s, bins=HIST_BINS, alpha=0.45, density=True,
                     label="Synthetic", color="darkorange")
        axes[i].set_title(c, fontsize=9)
        axes[i].tick_params(labelsize=7)
        if i == 0:
            axes[i].legend(fontsize=7)

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close(fig)


def plot_onehot_rates(real_df, syn_df, bin_cols, title, out_path):
    cols = [c for c in bin_cols if c in real_df.columns and c in syn_df.columns]
    if not cols:
        return
    real_rates = [real_df[c].fillna(0).mean() for c in cols]
    syn_rates  = [syn_df[c].fillna(0).mean()  for c in cols]

    # Only keep cols where at least one source has > 1% activation
    active = [(c, r, s) for c, r, s in zip(cols, real_rates, syn_rates) if max(r, s) > 0.01]
    if not active:
        return
    cols_a, real_a, syn_a = zip(*active)

    x = np.arange(len(cols_a))
    w = 0.35
    fig, ax = plt.subplots(figsize=(max(8, len(cols_a) * 0.55), 4))
    ax.bar(x - w/2, real_a, w, label="Real",      color="steelblue",  alpha=0.8)
    ax.bar(x + w/2, syn_a,  w, label="Synthetic", color="darkorange", alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(cols_a, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Activation rate")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.set_title(title)
    plt.tight_layout()
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close(fig)


print("Plot helpers defined.")

Plot helpers defined.


In [6]:
# ── Run balanced evaluation ──────────────────────────────────────────────────

balanced_results = {}   # {model: metrics_dict}

BALANCED_CONFIGS = {
    "ctgan": {
        "real_path": CTGAN_ROOT / "Balanced" / "train_input_with_label.csv",
        "syn_path":  CTGAN_ROOT / "Balanced" / "synthetic_ctgan_combined.csv",
        "out_dir":   OUT_ROOT / "ctgan" / "balanced",
    },
    "tabddpm": {
        "real_path": TABDDPM_ROOT / "Balanced" / "train_input_with_label.csv",
        "syn_path":  TABDDPM_ROOT / "Balanced" / "synthetic_tabddpm_combined.csv",
        "out_dir":   OUT_ROOT / "tabddpm" / "balanced",
    },
}

for model, cfg in BALANCED_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"BALANCED — {model.upper()}")
    print(f"{'='*60}")

    real_df = read_csv_safe(cfg["real_path"])
    syn_df  = read_csv_safe(cfg["syn_path"])

    if real_df is None or syn_df is None:
        print(f"  Skipping {model} balanced — data not available.")
        continue

    # Strip label from syn if absent
    real_feats = real_df.drop(columns=[LABEL_COL], errors="ignore")
    syn_feats  = syn_df.drop(columns=[LABEL_COL], errors="ignore")

    num_cols, bin_cols = detect_feature_types(real_feats)
    feat_cols = pick_numeric_features(real_feats, num_cols)

    metrics = evaluate_pair(real_feats, syn_feats, label="Balanced", feat_cols=feat_cols)
    balanced_results[model] = metrics

    out_dir = cfg["out_dir"]

    # Save metrics JSON
    with open(out_dir / "metrics.json", "w") as f:
        json.dump({k: v for k, v in metrics.items()
                   if isinstance(v, (int, float, str, dict, type(None)))}, f, indent=2)

    # Plots
    plot_distribution_grid(
        real_feats, syn_feats, feat_cols,
        f"{model.upper()} Balanced — Numeric feature distributions",
        out_dir / "distributions_numeric.png"
    )
    plot_correlation_comparison(
        real_feats, syn_feats, feat_cols,
        f"{model.upper()} Balanced — Correlation comparison",
        out_dir / "correlation_comparison.png"
    )
    plot_pca_overlay(
        real_feats, syn_feats, feat_cols,
        f"{model.upper()} Balanced — PCA projection (Real vs Synthetic)",
        out_dir / "pca_overlay.png"
    )
    plot_onehot_rates(
        real_feats, syn_feats, bin_cols,
        f"{model.upper()} Balanced — One-hot activation rates",
        out_dir / "onehot_rates.png"
    )

    # Summary printout
    print(f"  Real rows:       {metrics['real_rows']:>10,}")
    print(f"  Synthetic rows:  {metrics['syn_rows']:>10,}")
    print(f"  Avg KS:          {metrics['avg_ks']:.4f}  (lower = more similar)")
    print(f"  Avg Wasserstein: {metrics['avg_wasserstein']:.4f}")
    print(f"  Avg TVD binary:  {metrics['avg_tvd_binary']:.4f}")
    print(f"  Corr Frobenius:  {metrics['corr_frobenius']:.4f}")
    print(f"  PCA coverage:    {metrics['pca_coverage']:.4f}  (higher = better coverage)")
    print(f"  MMD:             {metrics['mmd']:.6f}  (lower = more similar)")
    print(f"  Outputs saved to: {out_dir}")


BALANCED — CTGAN
  Real rows:          160,000
  Synthetic rows:     160,000
  Avg KS:          0.4216  (lower = more similar)
  Avg Wasserstein: 0.0620
  Avg TVD binary:  0.0209
  Corr Frobenius:  1.5395
  PCA coverage:    0.8168  (higher = better coverage)
  MMD:             0.008627  (lower = more similar)
  Outputs saved to: augmentation_eval/ctgan/balanced

BALANCED — TABDDPM
  Real rows:          160,000
  Synthetic rows:     160,000
  Avg KS:          0.2659  (lower = more similar)
  Avg Wasserstein: 7.1296
  Avg TVD binary:  0.0285
  Corr Frobenius:  2.8928
  PCA coverage:    0.6829  (higher = better coverage)
  MMD:             0.219519  (lower = more similar)
  Outputs saved to: augmentation_eval/tabddpm/balanced


In [7]:
# ── Run per-class evaluation ──────────────────────────────────────────────────

perclass_results = defaultdict(dict)   # {model: {class_name: metrics_dict}}

PERCLASS_CONFIGS = {
    "ctgan": {
        "root":        CTGAN_ROOT,
        "syn_pattern": "synthetic_ctgan_unconditional.csv",
        "out_dir":     OUT_ROOT / "ctgan" / "perclass",
    },
    "tabddpm": {
        "root":        TABDDPM_ROOT,
        "syn_pattern": "synthetic_tabddpm_unconditional.csv",
        "out_dir":     OUT_ROOT / "tabddpm" / "perclass",
    },
}

for model, cfg in PERCLASS_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"PER-CLASS — {model.upper()}")
    print(f"{'='*60}")

    model_root = cfg["root"]
    syn_file   = cfg["syn_pattern"]
    out_dir    = cfg["out_dir"]

    class_dirs = sorted([
        d for d in model_root.iterdir()
        if d.is_dir() and d.name != "Balanced"
    ])

    for cls_dir in class_dirs:
        cls_name = cls_dir.name
        real_path = cls_dir / "train_input_with_label.csv"
        syn_path  = cls_dir / syn_file

        real_df = read_csv_safe(real_path)
        syn_df  = read_csv_safe(syn_path)

        if real_df is None or syn_df is None:
            print(f"  [{cls_name}] Skipping — data not available.")
            continue

        real_feats = real_df.drop(columns=[LABEL_COL], errors="ignore")
        syn_feats  = syn_df.drop(columns=[LABEL_COL],  errors="ignore")

        num_cols, bin_cols = detect_feature_types(real_feats)
        feat_cols = pick_numeric_features(real_feats, num_cols)

        metrics = evaluate_pair(real_feats, syn_feats, label=cls_name, feat_cols=feat_cols)
        perclass_results[model][cls_name] = metrics

        cls_out = out_dir / cls_name
        cls_out.mkdir(parents=True, exist_ok=True)

        with open(cls_out / "metrics.json", "w") as f:
            json.dump({k: v for k, v in metrics.items()
                       if isinstance(v, (int, float, str, dict, type(None)))}, f, indent=2)

        plot_distribution_grid(
            real_feats, syn_feats, feat_cols,
            f"{model.upper()} {cls_name} — Numeric distributions",
            cls_out / "distributions_numeric.png"
        )
        plot_correlation_comparison(
            real_feats, syn_feats, feat_cols,
            f"{model.upper()} {cls_name} — Correlation comparison",
            cls_out / "correlation_comparison.png"
        )
        plot_pca_overlay(
            real_feats, syn_feats, feat_cols,
            f"{model.upper()} {cls_name} — PCA overlay",
            cls_out / "pca_overlay.png"
        )
        plot_onehot_rates(
            real_feats, syn_feats, bin_cols,
            f"{model.upper()} {cls_name} — One-hot rates",
            cls_out / "onehot_rates.png"
        )

        print(f"  [{cls_name:16s}] KS={metrics['avg_ks']:.3f}  "
              f"WD={metrics['avg_wasserstein']:.3f}  "
              f"TVD={metrics['avg_tvd_binary']:.3f}  "
              f"Frob={metrics['corr_frobenius']:.3f}  "
              f"PCA_cov={metrics['pca_coverage']:.3f}  "
              f"MMD={metrics['mmd']:.5f}")


PER-CLASS — CTGAN
  [Benign          ] KS=0.463  WD=0.113  TVD=0.030  Frob=1.725  PCA_cov=0.831  MMD=0.01876
  [HTTPFlood       ] KS=0.346  WD=0.052  TVD=0.004  Frob=0.553  PCA_cov=0.886  MMD=0.00598
  [ICMPFlood       ] KS=0.597  WD=0.034  TVD=0.000  Frob=1.451  PCA_cov=0.953  MMD=0.00523
  [SYNFlood        ] KS=0.492  WD=0.114  TVD=0.003  Frob=0.746  PCA_cov=0.892  MMD=0.00642
  [SYNScan         ] KS=0.679  WD=0.051  TVD=0.005  Frob=3.136  PCA_cov=nan  MMD=0.00033
  [SlowrateDoS     ] KS=0.131  WD=0.061  TVD=0.003  Frob=0.768  PCA_cov=0.947  MMD=0.01445
  [TCPConnectScan  ] KS=0.785  WD=0.189  TVD=0.006  Frob=2.187  PCA_cov=nan  MMD=0.00175
  [UDPFlood        ] KS=0.318  WD=0.023  TVD=0.003  Frob=0.500  PCA_cov=0.821  MMD=0.00411
  [UDPScan         ] KS=0.656  WD=0.060  TVD=0.009  Frob=3.819  PCA_cov=nan  MMD=0.00670

PER-CLASS — TABDDPM
  [Benign          ] KS=0.327  WD=1.931  TVD=0.029  Frob=1.378  PCA_cov=0.839  MMD=0.48168
  [HTTPFlood       ] KS=0.483  WD=0.701  TVD=0.064  Frob

In [8]:
# ── Per-class summary heatmaps ────────────────────────────────────────────────

for model in ["ctgan", "tabddpm"]:
    cls_data = perclass_results.get(model, {})
    if not cls_data:
        continue

    classes = sorted(cls_data.keys())

    # ── KS heatmap per feature per class
    all_ks_feats = sorted(set(
        feat for d in cls_data.values() for feat in d["ks_per_feature"]
    ))
    if all_ks_feats:
        ks_mat = pd.DataFrame(
            {cls: [cls_data[cls]["ks_per_feature"].get(f, np.nan) for f in all_ks_feats]
             for cls in classes},
            index=all_ks_feats
        )
        fig, ax = plt.subplots(figsize=(max(8, len(classes)*1.1), max(5, len(all_ks_feats)*0.5)))
        sns.heatmap(ks_mat, ax=ax, cmap="YlOrRd", vmin=0, vmax=1,
                    annot=True, fmt=".2f", linewidths=0.3, cbar_kws={"label": "KS statistic"})
        ax.set_title(f"{model.upper()} Per-Class — KS Statistic per Feature")
        ax.set_xlabel("Attack Class")
        ax.set_ylabel("Feature")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        out_path = OUT_ROOT / model / "perclass" / "heatmap_ks_per_feature_per_class.png"
        fig.savefig(out_path, dpi=130, bbox_inches="tight")
        plt.close(fig)
        print(f"[{model}] KS heatmap saved: {out_path}")

    # ── Wasserstein heatmap per feature per class
    all_wd_feats = sorted(set(
        feat for d in cls_data.values() for feat in d["wasserstein_per_feature"]
    ))
    if all_wd_feats:
        wd_mat = pd.DataFrame(
            {cls: [cls_data[cls]["wasserstein_per_feature"].get(f, np.nan) for f in all_wd_feats]
             for cls in classes},
            index=all_wd_feats
        )
        fig, ax = plt.subplots(figsize=(max(8, len(classes)*1.1), max(5, len(all_wd_feats)*0.5)))
        sns.heatmap(wd_mat, ax=ax, cmap="Blues",
                    annot=True, fmt=".3f", linewidths=0.3, cbar_kws={"label": "Wasserstein distance"})
        ax.set_title(f"{model.upper()} Per-Class — Wasserstein Distance per Feature")
        ax.set_xlabel("Attack Class")
        ax.set_ylabel("Feature")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        out_path = OUT_ROOT / model / "perclass" / "heatmap_wasserstein_per_feature_per_class.png"
        fig.savefig(out_path, dpi=130, bbox_inches="tight")
        plt.close(fig)
        print(f"[{model}] Wasserstein heatmap saved: {out_path}")

    # ── TVD heatmap for binary features per class
    all_tvd_feats = sorted(set(
        feat for d in cls_data.values() for feat in d["tvd_per_binary_feature"]
    ))
    if all_tvd_feats:
        tvd_mat = pd.DataFrame(
            {cls: [cls_data[cls]["tvd_per_binary_feature"].get(f, np.nan) for f in all_tvd_feats]
             for cls in classes},
            index=all_tvd_feats
        )
        fig, ax = plt.subplots(figsize=(max(8, len(classes)*1.1), max(6, len(all_tvd_feats)*0.38)))
        sns.heatmap(tvd_mat, ax=ax, cmap="Oranges", vmin=0, vmax=0.5,
                    annot=True, fmt=".2f", linewidths=0.3, cbar_kws={"label": "TVD"})
        ax.set_title(f"{model.upper()} Per-Class — One-Hot TVD per Feature")
        ax.set_xlabel("Attack Class")
        ax.set_ylabel("Binary Feature")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        out_path = OUT_ROOT / model / "perclass" / "heatmap_tvd_binary_per_class.png"
        fig.savefig(out_path, dpi=130, bbox_inches="tight")
        plt.close(fig)
        print(f"[{model}] TVD heatmap saved: {out_path}")

[ctgan] KS heatmap saved: augmentation_eval/ctgan/perclass/heatmap_ks_per_feature_per_class.png
[ctgan] Wasserstein heatmap saved: augmentation_eval/ctgan/perclass/heatmap_wasserstein_per_feature_per_class.png
[ctgan] TVD heatmap saved: augmentation_eval/ctgan/perclass/heatmap_tvd_binary_per_class.png
[tabddpm] KS heatmap saved: augmentation_eval/tabddpm/perclass/heatmap_ks_per_feature_per_class.png
[tabddpm] Wasserstein heatmap saved: augmentation_eval/tabddpm/perclass/heatmap_wasserstein_per_feature_per_class.png
[tabddpm] TVD heatmap saved: augmentation_eval/tabddpm/perclass/heatmap_tvd_binary_per_class.png


In [9]:
# ── Summary: Balanced vs Per-Class per model, and CTGAN vs tabDDPM ───────────

SCALAR_METRICS = [
    "avg_ks", "avg_wasserstein", "avg_tvd_binary",
    "corr_frobenius", "pca_coverage", "mmd"
]
METRIC_LABELS = {
    "avg_ks":           "Avg KS (lower=better)",
    "avg_wasserstein":  "Avg Wasserstein (lower=better)",
    "avg_tvd_binary":   "Avg TVD binary (lower=better)",
    "corr_frobenius":   "Corr Frobenius dist (lower=better)",
    "pca_coverage":     "PCA coverage (higher=better)",
    "mmd":              "MMD (lower=better)",
}

# Build summary rows
summary_rows = []

# Balanced rows
for model, mets in balanced_results.items():
    row = {"model": model.upper(), "strategy": "Balanced", "class": "ALL"}
    for m in SCALAR_METRICS:
        row[m] = mets.get(m, np.nan)
    summary_rows.append(row)

# Per-class rows — also aggregate mean across classes
for model, cls_data in perclass_results.items():
    # Individual class rows
    for cls_name, mets in cls_data.items():
        row = {"model": model.upper(), "strategy": "PerClass", "class": cls_name}
        for m in SCALAR_METRICS:
            row[m] = mets.get(m, np.nan)
        summary_rows.append(row)

    # Aggregate mean across all per-class runs
    agg_row = {"model": model.upper(), "strategy": "PerClass", "class": "MEAN_ALL"}
    for m in SCALAR_METRICS:
        vals = [mets.get(m, np.nan) for mets in cls_data.values()]
        agg_row[m] = float(np.nanmean(vals)) if vals else np.nan
    summary_rows.append(agg_row)

summary_df = pd.DataFrame(summary_rows)
summary_csv = OUT_ROOT / "comparison" / "global_fidelity_summary.csv"
summary_df.to_csv(summary_csv, index=False)
print("Global summary CSV saved:", summary_csv)
print()
display(summary_df.set_index(["model", "strategy", "class"]))

Global summary CSV saved: augmentation_eval/comparison/global_fidelity_summary.csv



avg_ks  avg_wasserstein  avg_tvd_binary  \
model   strategy class                                                     
CTGAN   Balanced ALL             0.4216           0.0620          0.0209   
TABDDPM Balanced ALL             0.2659           7.1296          0.0285   
CTGAN   PerClass Benign          0.4631           0.1132          0.0304   
                 HTTPFlood       0.3465           0.0523          0.0038   
                 ICMPFlood       0.5966           0.0337          0.0000   
                 SYNFlood        0.4916           0.1139          0.0032   
                 SYNScan         0.6787           0.0505          0.0052   
                 SlowrateDoS     0.1305           0.0608          0.0028   
                 TCPConnectScan  0.7847           0.1891          0.0060   
                 UDPFlood        0.3183           0.0233          0.0028   
                 UDPScan         0.6556           0.0603          0.0086   
                 MEAN_ALL        0.4962           0.0775          0.0070   
TABDDPM PerClass Benign          0.3266           1.9315          0.0292   
                 HTTPFlood       0.4835           0.7015          0.0636   
                 ICMPFlood       0.0294           0.1063          0.0000   
                 SYNFlood        0.1208           0.4409          0.0086   
                 SYNScan         0.4269           1.8564          0.0850   
                 SlowrateDoS     0.0744           0.2264          0.0045   
                 TCPConnectScan  0.7297           3.5754          0.0706   
                 UDPFlood        0.4444           0.4722          0.0043   
                 UDPScan         0.3190          11.1009          0.0135   
                 MEAN_ALL        0.3283           2.2679          0.0310   

                                 corr_frobenius  pca_coverage    mmd  
model   strategy class                                                
CTGAN   Balanced ALL                     1.5395        0.8168 0.0086  
TABDDPM Balanced ALL                     2.8928        0.6829 0.2195  
CTGAN   PerClass Benign                  1.7255        0.8307 0.0188  
                 HTTPFlood               0.5531        0.8858 0.0060  
                 ICMPFlood               1.4507        0.9525 0.0052  
                 SYNFlood                0.7462        0.8919 0.0064  
                 SYNScan                 3.1355           NaN 0.0003  
                 SlowrateDoS             0.7683        0.9466 0.0144  
                 TCPConnectScan          2.1867           NaN 0.0018  
                 UDPFlood                0.5003        0.8209 0.0041  
                 UDPScan                 3.8190           NaN 0.0067  
                 MEAN_ALL                1.6539        0.8881 0.0071  
TABDDPM PerClass Benign                  1.3782        0.8392 0.4817  
                 HTTPFlood               4.4866        0.8299 0.1709  
                 ICMPFlood                  NaN           NaN    NaN  
                 SYNFlood                3.9688        0.9414 0.0379  
                 SYNScan                 1.9701           NaN 0.8563  
                 SlowrateDoS             1.9495        0.9246 0.0066  
                 TCPConnectScan          1.9320           NaN 0.8822  
                 UDPFlood                2.9832        0.8645 0.3137  
                 UDPScan                 1.9332           NaN 0.8661  
                 MEAN_ALL                2.5752        0.8799 0.4519

In [10]:
# ── Comparison plot: Balanced vs Per-Class (mean) per model ─────────────────

compare_rows = summary_df[
    (summary_df["strategy"] == "Balanced") |
    ((summary_df["strategy"] == "PerClass") & (summary_df["class"] == "MEAN_ALL"))
].copy()
compare_rows["config"] = compare_rows["model"] + " " + compare_rows["strategy"]

n_metrics = len(SCALAR_METRICS)
ncols = 3
nrows = (n_metrics + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4))
axes = np.array(axes).flatten()

colors = ["steelblue", "cornflowerblue", "darkorange", "sandybrown"]

for i, m in enumerate(SCALAR_METRICS):
    ax = axes[i]
    configs = compare_rows["config"].tolist()
    vals    = compare_rows[m].tolist()
    x = np.arange(len(configs))
    bars = ax.bar(x, vals, color=colors[:len(configs)], edgecolor="white", linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(configs, rotation=20, ha="right", fontsize=9)
    ax.set_title(METRIC_LABELS[m], fontsize=9)
    ax.set_ylabel(m, fontsize=8)
    for bar, v in zip(bars, vals):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                    f"{v:.4f}", ha="center", va="bottom", fontsize=7.5)

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

fig.suptitle("Fidelity Comparison: Balanced vs Per-Class (mean across classes)",
             fontsize=12, y=1.01)
plt.tight_layout()
out_path = OUT_ROOT / "comparison" / "balanced_vs_perclass_comparison.png"
fig.savefig(out_path, dpi=130, bbox_inches="tight")
plt.close(fig)
print("Comparison plot saved:", out_path)

Comparison plot saved: augmentation_eval/comparison/balanced_vs_perclass_comparison.png


In [11]:
# ── Comparison heatmap: all metrics x all configs ────────────────────────────

heat_data = compare_rows.set_index("config")[SCALAR_METRICS]

# Normalise each column 0..1 for visual comparison
heat_norm = heat_data.copy()
for col in heat_norm.columns:
    mn, mx = heat_norm[col].min(), heat_norm[col].max()
    if mx > mn:
        heat_norm[col] = (heat_norm[col] - mn) / (mx - mn)
    else:
        heat_norm[col] = 0.5

fig, ax = plt.subplots(figsize=(len(SCALAR_METRICS) * 1.8, len(heat_norm) * 1.0 + 1.5))
sns.heatmap(
    heat_norm,
    ax=ax,
    cmap="RdYlGn_r",   # red=high (worse for most metrics), green=low (better)
    vmin=0, vmax=1,
    annot=heat_data.round(4),  # show raw values
    fmt="",
    linewidths=0.5,
    cbar_kws={"label": "Normalised (0=best, 1=worst for most metrics)"}
)
ax.set_xticklabels([METRIC_LABELS[m].split(" (")[0] for m in SCALAR_METRICS],
                   rotation=25, ha="right", fontsize=9)
ax.set_title("Fidelity Summary Heatmap — All Configurations", fontsize=11)
plt.tight_layout()
out_path = OUT_ROOT / "comparison" / "fidelity_summary_heatmap.png"
fig.savefig(out_path, dpi=130, bbox_inches="tight")
plt.close(fig)
print("Summary heatmap saved:", out_path)

Summary heatmap saved: augmentation_eval/comparison/fidelity_summary_heatmap.png


In [12]:
# ── Per-class aggregate: CTGAN vs tabDDPM comparison per class ───────────────

common_classes = sorted(
    set(perclass_results.get("ctgan", {}).keys()) &
    set(perclass_results.get("tabddpm", {}).keys())
)

if common_classes:
    for m in ["avg_ks", "avg_tvd_binary", "pca_coverage", "mmd"]:
        ctgan_vals   = [perclass_results["ctgan"][c].get(m, np.nan)   for c in common_classes]
        tabddpm_vals = [perclass_results["tabddpm"][c].get(m, np.nan) for c in common_classes]

        x = np.arange(len(common_classes))
        w = 0.35
        fig, ax = plt.subplots(figsize=(max(9, len(common_classes)*1.1), 4))
        ax.bar(x - w/2, ctgan_vals,   w, label="CTGAN",   color="steelblue",  alpha=0.85)
        ax.bar(x + w/2, tabddpm_vals, w, label="tabDDPM", color="darkorange", alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(common_classes, rotation=30, ha="right", fontsize=9)
        ax.set_ylabel(m)
        ax.set_title(f"Per-Class Comparison: CTGAN vs tabDDPM — {METRIC_LABELS[m]}")
        ax.legend()
        plt.tight_layout()
        out_path = OUT_ROOT / "comparison" / f"perclass_ctgan_vs_tabddpm_{m}.png"
        fig.savefig(out_path, dpi=130, bbox_inches="tight")
        plt.close(fig)
        print(f"Per-class comparison plot saved: {out_path}")

    # Combined radar / spider chart summary
    print("\n--- Per-class CTGAN vs tabDDPM scalar summary ---")
    summary_pc = pd.DataFrame(
        index=common_classes,
        data={
            f"ctgan_{m}": [perclass_results["ctgan"][c].get(m, np.nan) for c in common_classes]
            for m in SCALAR_METRICS
        } | {
            f"tabddpm_{m}": [perclass_results["tabddpm"][c].get(m, np.nan) for c in common_classes]
            for m in SCALAR_METRICS
        }
    )
    summary_pc.to_csv(OUT_ROOT / "comparison" / "perclass_ctgan_vs_tabddpm_all_metrics.csv")
    display(summary_pc)

Per-class comparison plot saved: augmentation_eval/comparison/perclass_ctgan_vs_tabddpm_avg_ks.png
Per-class comparison plot saved: augmentation_eval/comparison/perclass_ctgan_vs_tabddpm_avg_tvd_binary.png
Per-class comparison plot saved: augmentation_eval/comparison/perclass_ctgan_vs_tabddpm_pca_coverage.png
Per-class comparison plot saved: augmentation_eval/comparison/perclass_ctgan_vs_tabddpm_mmd.png

--- Per-class CTGAN vs tabDDPM scalar summary ---


,ctgan_avg_ks,ctgan_avg_wasserstein,ctgan_avg_tvd_binary,ctgan_corr_frobenius,ctgan_pca_coverage,ctgan_mmd,tabddpm_avg_ks,tabddpm_avg_wasserstein,tabddpm_avg_tvd_binary,tabddpm_corr_frobenius,tabddpm_pca_coverage,tabddpm_mmd
Benign,0.4631,0.1132,0.0304,1.7255,0.8307,0.0188,0.3266,1.9315,0.0292,1.3782,0.8392,0.4817
HTTPFlood,0.3465,0.0523,0.0038,0.5531,0.8858,0.0060,0.4835,0.7015,0.0636,4.4866,0.8299,0.1709
ICMPFlood,0.5966,0.0337,0.0000,1.4507,0.9525,0.0052,0.0294,0.1063,0.0000,NaN,NaN,NaN
SYNFlood,0.4916,0.1139,0.0032,0.7462,0.8919,0.0064,0.1208,0.4409,0.0086,3.9688,0.9414,0.0379
SYNScan,0.6787,0.0505,0.0052,3.1355,NaN,0.0003,0.4269,1.8564,0.0850,1.9701,NaN,0.8563
SlowrateDoS,0.1305,0.0608,0.0028,0.7683,0.9466,0.0144,0.0744,0.2264,0.0045,1.9495,0.9246,0.0066
TCPConnectScan,0.7847,0.1891,0.0060,2.1867,NaN,0.0018,0.7297,3.5754,0.0706,1.9320,NaN,0.8822
UDPFlood,0.3183,0.0233,0.0028,0.5003,0.8209,0.0041,0.4444,0.4722,0.0043,2.9832,0.8645,0.3137
UDPScan,0.6556,0.0603,0.0086,3.8190,NaN,0.0067,0.3190,11.1009,0.0135,1.9332,NaN,0.8661


In [13]:
# ── Final printed summary ─────────────────────────────────────────────────────

print("\n" + "="*70)
print("AUGMENTATION FIDELITY EVALUATION — FINAL SUMMARY")
print("="*70)

highlight = summary_df[
    summary_df["class"].isin(["ALL", "MEAN_ALL"])
].copy()

for _, row in highlight.iterrows():
    print(f"\n  {row['model']:8s} {row['strategy']:10s}")
    for m in SCALAR_METRICS:
        v = row[m]
        direction = "↓ lower=better" if m != "pca_coverage" else "↑ higher=better"
        print(f"    {METRIC_LABELS[m]:42s} {v:.5f}   {direction}")

print("\n" + "="*70)
print(f"All outputs saved under: {OUT_ROOT.resolve()}")
print("="*70)


AUGMENTATION FIDELITY EVALUATION — FINAL SUMMARY

  CTGAN    Balanced  
    Avg KS (lower=better)                      0.42161   ↓ lower=better
    Avg Wasserstein (lower=better)             0.06201   ↓ lower=better
    Avg TVD binary (lower=better)              0.02090   ↓ lower=better
    Corr Frobenius dist (lower=better)         1.53947   ↓ lower=better
    PCA coverage (higher=better)               0.81679   ↑ higher=better
    MMD (lower=better)                         0.00863   ↓ lower=better

  TABDDPM  Balanced  
    Avg KS (lower=better)                      0.26588   ↓ lower=better
    Avg Wasserstein (lower=better)             7.12962   ↓ lower=better
    Avg TVD binary (lower=better)              0.02852   ↓ lower=better
    Corr Frobenius dist (lower=better)         2.89281   ↓ lower=better
    PCA coverage (higher=better)               0.68294   ↑ higher=better
    MMD (lower=better)                         0.21952   ↓ lower=better

  CTGAN    PerClass  
    Avg KS (low

---
## Winner Determination: CTGAN vs TabDDPM

This section computes a structured, metric-weighted verdict on which generative model
produced higher-fidelity synthetic data, both overall and per augmentation strategy.

### Scoring logic
Each metric is directional (lower is better for KS, Wasserstein, TVD, Frobenius, MMD;
higher is better for PCA coverage). For each metric, the model with the better value
receives 1 point; ties receive 0.5 each. Metrics are optionally weighted by importance.
A detailed verdict table is printed and saved, along with a visual scoreboard.


In [14]:
# ── Winner Determination: CTGAN vs tabDDPM ────────────────────────────────────

# Metric properties: (higher_is_better, weight)
# Weights reflect importance in the tabular data fidelity literature:
#   MMD and KS are the primary distributional metrics,
#   PCA coverage and Frobenius are secondary structural metrics,
#   TVD and Wasserstein are supporting marginal metrics.
METRIC_CONFIG = {
    "avg_ks":           (False, 2.0),   # primary distributional
    "mmd":              (False, 2.0),   # primary global distance
    "pca_coverage":     (True,  1.5),   # structural coverage
    "corr_frobenius":   (False, 1.5),   # correlation structure
    "avg_wasserstein":  (False, 1.0),   # marginal support
    "avg_tvd_binary":   (False, 1.0),   # binary feature fidelity
}

STRATEGIES = ["Balanced", "PerClass_mean"]

def extract_strategy_metrics(model, strategy):
    """Pull scalar metrics for a given model and strategy."""
    if strategy == "Balanced":
        return balanced_results.get(model, {})
    elif strategy == "PerClass_mean":
        cls_data = perclass_results.get(model, {})
        if not cls_data:
            return {}
        agg = {}
        for m in SCALAR_METRICS:
            vals = [v.get(m, np.nan) for v in cls_data.values()]
            agg[m] = float(np.nanmean(vals)) if vals else np.nan
        return agg
    return {}


def score_pair(m1, m2, higher_is_better, weight):
    """Return (score_model1, score_model2) for one metric."""
    if np.isnan(m1) or np.isnan(m2):
        return 0.0, 0.0
    if higher_is_better:
        if m1 > m2 + 1e-9:   return weight, 0.0
        if m2 > m1 + 1e-9:   return 0.0, weight
        return weight / 2, weight / 2
    else:
        if m1 < m2 - 1e-9:   return weight, 0.0
        if m2 < m1 - 1e-9:   return 0.0, weight
        return weight / 2, weight / 2


verdict_rows = []

for strategy in STRATEGIES:
    ctgan_m   = extract_strategy_metrics("ctgan",   strategy)
    tabddpm_m = extract_strategy_metrics("tabddpm", strategy)

    if not ctgan_m and not tabddpm_m:
        print(f"[{strategy}] No data available — skipping.")
        continue

    total_ctgan = total_tabddpm = 0.0
    detail_rows = []

    for metric, (higher_is_better, weight) in METRIC_CONFIG.items():
        v_ctgan   = ctgan_m.get(metric, np.nan)
        v_tabddpm = tabddpm_m.get(metric, np.nan)
        s_c, s_t  = score_pair(v_ctgan, v_tabddpm, higher_is_better, weight)
        total_ctgan   += s_c
        total_tabddpm += s_t

        direction = "higher=better" if higher_is_better else "lower=better"
        winner_metric = "CTGAN" if s_c > s_t else ("tabDDPM" if s_t > s_c else "TIE")
        detail_rows.append({
            "strategy":    strategy,
            "metric":      metric,
            "direction":   direction,
            "weight":      weight,
            "ctgan_value": round(v_ctgan, 6) if not np.isnan(v_ctgan) else None,
            "tabddpm_value": round(v_tabddpm, 6) if not np.isnan(v_tabddpm) else None,
            "ctgan_pts":   s_c,
            "tabddpm_pts": s_t,
            "metric_winner": winner_metric,
        })

    max_pts = sum(w for _, (_, w) in METRIC_CONFIG.items())
    overall_winner = "CTGAN" if total_ctgan > total_tabddpm else (
        "tabDDPM" if total_tabddpm > total_ctgan else "TIE")

    print(f"\n{'='*65}")
    print(f"  STRATEGY: {strategy}")
    print(f"{'='*65}")
    print(f"  {'Metric':<28} {'CTGAN':>10} {'tabDDPM':>10} {'Pts C':>6} {'Pts T':>6} {'Winner':<10}")
    print(f"  {'-'*63}")
    for dr in detail_rows:
        vc = f"{dr['ctgan_value']:.5f}"   if dr['ctgan_value']   is not None else "N/A"
        vt = f"{dr['tabddpm_value']:.5f}" if dr['tabddpm_value'] is not None else "N/A"
        print(f"  {dr['metric']:<28} {vc:>10} {vt:>10} {dr['ctgan_pts']:>6.1f} {dr['tabddpm_pts']:>6.1f} {dr['metric_winner']:<10}")
    print(f"  {'-'*63}")
    print(f"  {'TOTAL (weighted)':28} {total_ctgan:>10.2f} {total_tabddpm:>10.2f}  (max={max_pts:.1f})")
    print(f"\n  >>> WINNER [{strategy}]: {overall_winner} ({total_ctgan:.2f} vs {total_tabddpm:.2f} pts)")

    verdict_rows.extend(detail_rows)
    verdict_rows.append({
        "strategy":      strategy, "metric": "TOTAL",
        "direction":     "",       "weight":  max_pts,
        "ctgan_value":   round(total_ctgan,   4),
        "tabddpm_value": round(total_tabddpm, 4),
        "ctgan_pts":     total_ctgan,
        "tabddpm_pts":   total_tabddpm,
        "metric_winner": overall_winner,
    })

verdict_df = pd.DataFrame(verdict_rows)
verdict_csv = OUT_ROOT / "comparison" / "winner_verdict.csv"
verdict_df.to_csv(verdict_csv, index=False)
print(f"\nVerdict table saved: {verdict_csv}")


  STRATEGY: Balanced
  Metric                            CTGAN    tabDDPM  Pts C  Pts T Winner    
  ---------------------------------------------------------------
  avg_ks                          0.42161    0.26588    0.0    2.0 tabDDPM   
  mmd                             0.00863    0.21952    2.0    0.0 CTGAN     
  pca_coverage                    0.81679    0.68294    1.5    0.0 CTGAN     
  corr_frobenius                  1.53947    2.89281    1.5    0.0 CTGAN     
  avg_wasserstein                 0.06201    7.12962    1.0    0.0 CTGAN     
  avg_tvd_binary                  0.02090    0.02852    1.0    0.0 CTGAN     
  ---------------------------------------------------------------
  TOTAL (weighted)                   7.00       2.00  (max=9.0)

  >>> WINNER [Balanced]: CTGAN (7.00 vs 2.00 pts)

  STRATEGY: PerClass_mean
  Metric                            CTGAN    tabDDPM  Pts C  Pts T Winner    
  ---------------------------------------------------------------
  avg_ks      

In [15]:
# ── Scoreboard visualisation ──────────────────────────────────────────────────

fig, axes = plt.subplots(1, len(STRATEGIES), figsize=(6 * len(STRATEGIES), 5.5))
if len(STRATEGIES) == 1:
    axes = [axes]

palette = {"CTGAN": "steelblue", "tabDDPM": "darkorange", "TIE": "grey"}

for ax, strategy in zip(axes, STRATEGIES):
    sub = verdict_df[
        (verdict_df["strategy"] == strategy) & (verdict_df["metric"] != "TOTAL")
    ].copy()
    if sub.empty:
        ax.axis("off")
        continue

    metrics_here = sub["metric"].tolist()
    ctgan_pts    = sub["ctgan_pts"].tolist()
    tabddpm_pts  = sub["tabddpm_pts"].tolist()

    x = np.arange(len(metrics_here))
    w = 0.35
    b1 = ax.bar(x - w/2, ctgan_pts,   w, label="CTGAN",   color="steelblue",  alpha=0.85)
    b2 = ax.bar(x + w/2, tabddpm_pts, w, label="tabDDPM", color="darkorange", alpha=0.85)

    # Annotate total scores
    total_row = verdict_df[
        (verdict_df["strategy"] == strategy) & (verdict_df["metric"] == "TOTAL")
    ]
    if not total_row.empty:
        tc = total_row.iloc[0]["ctgan_value"]
        tt = total_row.iloc[0]["tabddpm_value"]
        winner = total_row.iloc[0]["metric_winner"]
        ax.set_title(
            f"{strategy}\nCTGAN {tc:.2f} pts  |  tabDDPM {tt:.2f} pts\n"
            f"Winner: {winner}",
            fontsize=10
        )

    ax.set_xticks(x)
    ax.set_xticklabels(metrics_here, rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("Weighted points won", fontsize=9)
    ax.set_ylim(0, max(ctgan_pts + tabddpm_pts) * 1.3 + 0.1)
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("CTGAN vs tabDDPM — Weighted Fidelity Scoreboard", fontsize=13, y=1.01)
plt.tight_layout()
out_path = OUT_ROOT / "comparison" / "winner_scoreboard.png"
fig.savefig(out_path, dpi=130, bbox_inches="tight")
plt.close(fig)
print("Scoreboard saved:", out_path)

Scoreboard saved: augmentation_eval/comparison/winner_scoreboard.png


In [16]:
# ── Radar chart: per-metric fidelity profile CTGAN vs tabDDPM ────────────────
# Normalise each metric 0..1 where 1 = best possible fidelity

import matplotlib.patches as mpatches
from matplotlib.path import Path as MPath

def radar_chart(categories, values_a, values_b, label_a, label_b, title, out_path):
    N = len(categories)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    angles += angles[:1]

    vals_a = list(values_a) + [values_a[0]]
    vals_b = list(values_b) + [values_b[0]]

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    ax.plot(angles, vals_a, "o-", color="steelblue",  linewidth=2, label=label_a)
    ax.fill(angles, vals_a, alpha=0.20, color="steelblue")
    ax.plot(angles, vals_b, "s-", color="darkorange", linewidth=2, label=label_b)
    ax.fill(angles, vals_b, alpha=0.15, color="darkorange")

    ax.set_thetagrids(np.degrees(angles[:-1]), categories, fontsize=9)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["0.25", "0.5", "0.75", "1.0"], fontsize=7)
    ax.set_title(title, fontsize=11, pad=15)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=9)
    plt.tight_layout()
    fig.savefig(out_path, dpi=130, bbox_inches="tight")
    plt.close(fig)


for strategy in STRATEGIES:
    ctgan_m   = extract_strategy_metrics("ctgan",   strategy)
    tabddpm_m = extract_strategy_metrics("tabddpm", strategy)
    if not ctgan_m or not tabddpm_m:
        continue

    # Normalise: for each metric collect both values, rescale so best=1, worst=0
    cats, c_vals, t_vals = [], [], []
    for m, (higher_is_better, _) in METRIC_CONFIG.items():
        vc = ctgan_m.get(m, np.nan)
        vt = tabddpm_m.get(m, np.nan)
        if np.isnan(vc) or np.isnan(vt):
            continue
        lo, hi = min(vc, vt), max(vc, vt)
        rng = hi - lo if hi != lo else 1.0
        # Normalise to [0,1] where 1 = best
        if higher_is_better:
            nc, nt = (vc - lo) / rng, (vt - lo) / rng
        else:
            nc, nt = (hi - vc) / rng, (hi - vt) / rng
        # If both equal, both get 0.5
        if lo == hi:
            nc, nt = 0.5, 0.5
        cats.append(m.replace("avg_", "").replace("_", "\n"))
        c_vals.append(nc)
        t_vals.append(nt)

    if cats:
        radar_chart(
            cats, c_vals, t_vals,
            label_a="CTGAN", label_b="tabDDPM",
            title=f"Fidelity Profile ({strategy})\n1 = better fidelity",
            out_path=OUT_ROOT / "comparison" / f"radar_{strategy.lower()}.png"
        )
        print(f"Radar chart saved for {strategy}")

Radar chart saved for Balanced
Radar chart saved for PerClass_mean


In [17]:
# ── Print final overall verdict ───────────────────────────────────────────────

print("\n" + "#" * 65)
print("  FINAL VERDICT: CTGAN vs tabDDPM — WHO DID A BETTER JOB?")
print("#" * 65)

totals = verdict_df[verdict_df["metric"] == "TOTAL"].copy()

ctgan_wins   = (totals["metric_winner"] == "CTGAN").sum()
tabddpm_wins = (totals["metric_winner"] == "tabDDPM").sum()
ties         = (totals["metric_winner"] == "TIE").sum()

print(f"\n  Strategies evaluated: {len(totals)}")
print(f"  CTGAN wins:   {ctgan_wins}")
print(f"  tabDDPM wins: {tabddpm_wins}")
print(f"  Ties:         {ties}")

for _, row in totals.iterrows():
    print(f"\n  [{row['strategy']}]  CTGAN {row['ctgan_value']:.2f} pts  vs  "
          f"tabDDPM {row['tabddpm_value']:.2f} pts  =>  Winner: {row['metric_winner']}")

# Overall weighted score across all strategies
overall_ctgan   = totals["ctgan_value"].sum()
overall_tabddpm = totals["tabddpm_value"].sum()
overall_winner  = "CTGAN" if overall_ctgan > overall_tabddpm else (
    "tabDDPM" if overall_tabddpm > overall_ctgan else "TIE")

print(f"\n  Combined score across all strategies:")
print(f"    CTGAN:   {overall_ctgan:.2f} pts")
print(f"    tabDDPM: {overall_tabddpm:.2f} pts")
print(f"\n  {'='*55}")
print(f"  OVERALL WINNER: {overall_winner}")
print(f"  {'='*55}")
print()
print("  Interpretation:")
print("  - A lower KS / Wasserstein / TVD / MMD indicates the synthetic")
print("    marginal and joint distributions are closer to the real data.")
print("  - A lower Frobenius distance means the correlation structure")
print("    of real features is better preserved in the synthetic data.")
print("  - A higher PCA coverage means the synthetic samples cover a")
print("    larger fraction of the real data manifold.")
print("  - Weights: MMD and KS x2.0 (primary); PCA coverage and Corr x1.5;")
print("    Wasserstein and TVD x1.0 (supporting).")
print("\n  All outputs saved under:", str(OUT_ROOT.resolve()))


#################################################################
  FINAL VERDICT: CTGAN vs tabDDPM — WHO DID A BETTER JOB?
#################################################################

  Strategies evaluated: 2
  CTGAN wins:   2
  tabDDPM wins: 0
  Ties:         0

  [Balanced]  CTGAN 7.00 pts  vs  tabDDPM 2.00 pts  =>  Winner: CTGAN

  [PerClass_mean]  CTGAN 7.00 pts  vs  tabDDPM 2.00 pts  =>  Winner: CTGAN

  Combined score across all strategies:
    CTGAN:   14.00 pts
    tabDDPM: 4.00 pts

  OVERALL WINNER: CTGAN

  Interpretation:
  - A lower KS / Wasserstein / TVD / MMD indicates the synthetic
    marginal and joint distributions are closer to the real data.
  - A lower Frobenius distance means the correlation structure
    of real features is better preserved in the synthetic data.
  - A higher PCA coverage means the synthetic samples cover a
    larger fraction of the real data manifold.
  - Weights: MMD and KS x2.0 (primary); PCA coverage and Corr x1.5;
    Wasserstein 